In [28]:
from dotenv import load_dotenv
import os
load_dotenv()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, to_date, year, month, quarter, when,
    sum as F_sum, count, countDistinct, avg
)

from pyspark.sql.functions import col, sum as spark_sum

# Preprocessing

## Setup

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("midterm") \
    .master("local[*]") \
    .getOrCreate()

25/11/26 12:12:22 WARN Utils: Your hostname, Lianas-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.20.123 instead (on interface en0)
25/11/26 12:12:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/26 12:12:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


These are normal on macOS and don’t affect Spark.
Macs don’t have native Hadoop libraries, so Spark uses its built-in Java version, which works fine. The hostname warning is also expected when running Spark locally.

So everything is working correctly.

In [3]:
tx_path = os.getenv("TX_PATH")
art_path = os.getenv("ART_PATH")

In [4]:
transactions = spark.read.csv(tx_path, header=True, inferSchema=True)
articles = spark.read.csv(art_path, header=True, inferSchema=True)

In [5]:
transactions.printSchema()

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [6]:
articles.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

In [7]:
transactions_count = transactions.count()
transactions_count

336078

In [8]:
articles_count = articles.count()
articles_count

105542

In [9]:
transactions.show(10)

+----------+--------------------+----------+------------------+----------------+
|     t_dat|         customer_id|article_id|             price|sales_channel_id|
+----------+--------------------+----------+------------------+----------------+
|2019-07-24|192e9ad7f0c05d89e...| 692721005|0.0121864406779661|               2|
|2019-04-07|70c1ee207c64a6523...| 599502013|0.0508305084745762|               2|
|2019-05-21|e29656435a0c04ef1...| 737260001|0.0254067796610169|               2|
|2019-03-29|13d1fd878959e117e...| 717251003|0.0169322033898305|               2|
|2019-07-05|58ddbcfa96eab2b3a...| 795675003| 0.008457627118644|               1|
|2019-06-04|eedec5ac3f7a88f41...| 742092001|0.0254067796610169|               2|
|2018-10-05|3de91e932fd943b2d...| 539197011|0.0135423728813559|               2|
|2019-06-04|810ed6f2725876b9d...| 372860002|0.0135423728813559|               1|
|2018-10-12|4de96a646f7030e83...| 645709001| 0.020322033898305|               2|
|2019-06-08|7d17548a65f00b2a

In [10]:
articles.show(10)

+----------+------------+--------------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|           prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|      index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+--------------------+----

In [15]:
transactions.select("customer_id").distinct().count()

230688

In [16]:
transactions.select("article_id").distinct().count()

41405

In [17]:
articles.select("article_id").distinct().count()

105542

In [18]:
transactions.describe("price").show()

+-------+-------------------+
|summary|              price|
+-------+-------------------+
|  count|             336078|
|   mean| 0.0274985190080482|
| stddev|0.01921244097171475|
|    min|  1.864406779661E-4|
|    max| 0.5067796610169492|
+-------+-------------------+



In [19]:
transactions.groupBy("sales_channel_id").count().show()

+----------------+------+
|sales_channel_id| count|
+----------------+------+
|               1|103464|
|               2|232614|
+----------------+------+



## Duplicate Check

In [20]:
transactions.count(), transactions.dropDuplicates().count()

(336078, 335151)

In [21]:
dupes = (
    transactions
    .groupBy(transactions.columns)
    .count()
    .filter("count > 1")
)

dupes.count()

895

In [22]:
dupes.show(truncate=False)

+----------+----------------------------------------------------------------+----------+------------------+----------------+-----+
|t_dat     |customer_id                                                     |article_id|price             |sales_channel_id|count|
+----------+----------------------------------------------------------------+----------+------------------+----------------+-----+
|2019-08-25|c025ac2ada70d4b94fa156ecf0ad948dc1791d74b8a194e654bd3d91a19a2a02|781683005 |0.0423559322033898|2               |2    |
|2018-12-13|97a788e36864a5634c518ad1b1ddc87da50169cd11b625f01490216f94a9042d|524825011 |0.0423559322033898|2               |2    |
|2019-05-16|e62b39733d3c123a32d62118df63317eb5b77c4b315e22151a98ce0264326bf9|722437001 |0.0220169491525423|2               |2    |
|2019-01-09|98c5c4ee85fab57b4fcf0b9b1055e8065fb454cdd92aa22929ab55e1d363475a|655248001 |0.0118474576271186|2               |2    |
|2019-04-06|7ba2cdcb79d4596615e1bbeaec8bf78328a64aaa7d166e9ec48558839ba19bdb|766346

In [23]:
transactions_clean = transactions.dropDuplicates()
transactions.count(), transactions_clean.count()

(336078, 335151)

Before analysis, I checked whether the transactions dataset contained fully identical duplicate rows.
Using transactions.count() and transactions.dropDuplicates().count(), I found:
- Original rows: 336,078
- Unique rows: 335,151
- Exact duplicates removed: 927

The duplicated rows were identical across all fields (t_dat, customer_id, article_id, price, sales_channel_id).
These duplicates are almost certainly caused by data export/ETL repetition, not by customers buying the same item twice at the same moment.

To avoid artificially inflating:
- number of transactions
- revenue
- customer-level metrics

I removed these duplicates using dropDuplicates().

## Nulls check

In [24]:
transactions.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in transactions.columns]).show()

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+



In [25]:
articles.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in articles.columns]).show()

+----------+------------+---------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------+--------------+----------------+----------+------------+----------------+------------------+-----------+
|article_id|product_code|prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|index_name|index_group_no|index_group_name|section_no|section_name|garment_group_no|garment_group_name|detail_desc|
+----------+------------+---------+---------------+-----------------+------------------+-----------------------+------

Only the column `detail_desc` contains missing values (416 rows).
This field is a free-text description used for product marketing and does not affect
any analytical tasks such as trend analysis, product grouping, or revenue estimation.

In [26]:
missing_articles = transactions \
    .join(articles, "article_id", "left_anti")

missing_articles.count()

0

## Validate Date Range

In [29]:
transactions.agg(
    F.min("t_dat").alias("min_date"),
    F.max("t_dat").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2018-09-20|2019-09-19|
+----------+----------+



## Feature Engineering

In [30]:
transactions = transactions \
    .withColumn("year", F.year("t_dat")) \
    .withColumn("month", F.month("t_dat"))

In [31]:
transactions = transactions.withColumn(
    "season",
    F.when(F.col("month").isin(12,1,2), "Winter")
     .when(F.col("month").isin(3,4,5), "Spring")
     .when(F.col("month").isin(6,7,8), "Summer")
     .otherwise("Fall")
)

# Task 1

## 1.a - Product Trends by Season

In [32]:
trx_joined = transactions.join(articles, on="article_id", how="left")

In [33]:
trx_joined.filter(F.col("prod_name").isNull()).count()

0

### Seasonal Sales Summary
I calculate the number of transactions and total revenue in each season.

In [34]:
season_summary = trx_joined.groupBy("season").agg(
    F.count("*").alias("transaction_count"),
    F.sum("price").alias("total_revenue")
).orderBy("season")

season_summary.show()

+------+-----------------+------------------+
|season|transaction_count|     total_revenue|
+------+-----------------+------------------+
|  Fall|            79526| 2441.777796610165|
|Spring|            85977| 2459.363118644083|
|Summer|            99222| 2387.082881355948|
|Winter|            71353|1953.4234745762828|
+------+-----------------+------------------+



**Findings:**
- **Summer** has the highest number of transactions (~99K) but not the highest revenue.
- **Spring** generates the highest revenue (~2459), despite fewer transactions than Summer.
- **Fall** is stable in both sales volume and revenue.
- **Winter** has the lowest activity in both metrics.

Possible interpretation:
- Winter has fewer purchases and lower spending per item.
- Spring and Summer appear to be the peak shopping seasons.

### Top Product Groups per Season
This section identifies which product groups are popular in each season.

In [35]:
season_groups = trx_joined.groupBy("season", "product_group_name") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", F.desc("count"))

season_groups.show(50, truncate=False)

+------+-------------------+-----+
|season|product_group_name |count|
+------+-------------------+-----+
|Fall  |Garment Upper body |38625|
|Fall  |Garment Lower body |18046|
|Fall  |Garment Full body  |5972 |
|Fall  |Underwear          |5523 |
|Fall  |Accessories        |4514 |
|Fall  |Socks & Tights     |2228 |
|Fall  |Shoes              |1932 |
|Fall  |Swimwear           |1576 |
|Fall  |Nightwear          |1035 |
|Fall  |Unknown            |48   |
|Fall  |Bags               |10   |
|Fall  |Items              |7    |
|Fall  |Cosmetic           |7    |
|Fall  |Underwear/nightwear|2    |
|Fall  |Interior textile   |1    |
|Spring|Garment Upper body |31380|
|Spring|Garment Lower body |19326|
|Spring|Garment Full body  |10568|
|Spring|Swimwear           |10305|
|Spring|Underwear          |6033 |
|Spring|Accessories        |3784 |
|Spring|Shoes              |2391 |
|Spring|Socks & Tights     |1510 |
|Spring|Nightwear          |554  |
|Spring|Unknown            |119  |
|Spring|Items       

### Top Departments per Season
Shows which parts of the store have seasonal peaks.

In [36]:
season_departments = trx_joined.groupBy("season", "department_name") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", F.desc("count"))

season_departments.show(50, truncate=False)

+------+-----------------------+-----+
|season|department_name        |count|
+------+-----------------------+-----+
|Fall  |Knitwear               |7244 |
|Fall  |Trouser                |4787 |
|Fall  |Blouse                 |3748 |
|Fall  |Jersey Basic           |3205 |
|Fall  |Jersey                 |3042 |
|Fall  |Basic 1                |2781 |
|Fall  |Expressive Lingerie    |2752 |
|Fall  |Tops Knitwear          |2430 |
|Fall  |Trousers               |2172 |
|Fall  |Denim Trousers         |2037 |
|Fall  |Jersey fancy           |1986 |
|Fall  |Tops Fancy Jersey      |1771 |
|Fall  |Outwear                |1511 |
|Fall  |Ladies Sport Bras      |1495 |
|Fall  |Dress                  |1486 |
|Fall  |Swimwear               |1456 |
|Fall  |Dresses                |1141 |
|Fall  |Tights basic           |1111 |
|Fall  |Casual Lingerie        |1100 |
|Fall  |Tops Woven             |1036 |
|Fall  |Ladies Sport Bottoms   |1028 |
|Fall  |Knitwear Basic         |933  |
|Fall  |Skirt            

### Top Product Types per Season
Here we analyze the most frequently purchased product types in each season.

In [37]:
season_types = trx_joined.groupBy("season", "product_type_name") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", F.desc("count"))

season_types.show(50, truncate=False)

+------+-----------------+-----+
|season|product_type_name|count|
+------+-----------------+-----+
|Fall  |Sweater          |14025|
|Fall  |Trousers         |12796|
|Fall  |Dress            |5421 |
|Fall  |T-shirt          |4562 |
|Fall  |Blouse           |3835 |
|Fall  |Top              |3519 |
|Fall  |Bra              |2759 |
|Fall  |Leggings/Tights  |2331 |
|Fall  |Underwear bottom |2297 |
|Fall  |Vest top         |2278 |
|Fall  |Jacket           |2276 |
|Fall  |Shirt            |2219 |
|Fall  |Skirt            |2121 |
|Fall  |Hoodie           |2046 |
|Fall  |Cardigan         |1485 |
|Fall  |Socks            |1346 |
|Fall  |Blazer           |1191 |
|Fall  |Boots            |881  |
|Fall  |Underwear Tights |879  |
|Fall  |Shorts           |774  |
|Fall  |Scarf            |751  |
|Fall  |Swimwear bottom  |640  |
|Fall  |Coat             |629  |
|Fall  |Pyjama set       |627  |
|Fall  |Bikini top       |615  |
|Fall  |Bag              |610  |
|Fall  |Hat/beanie       |473  |
|Fall  |Be

### Price difference by season
This section compares the average selling price across seasons.

In [38]:
season_price = trx_joined.groupBy("season") \
    .agg(
        F.mean("price").alias("avg_price"),
        F.expr("percentile(price, 0.5)").alias("median_price")
    )

season_price.show()

+------+--------------------+------------------+
|season|           avg_price|      median_price|
+------+--------------------+------------------+
|Spring| 0.02860489571215654|0.0254067796610169|
|Summer| 0.02405800005397944| 0.021593220338983|
|  Fall| 0.03070414451387175|0.0254067796610169|
|Winter|0.027376893397282283|0.0238474576271186|
+------+--------------------+------------------+



## 1.b - Product Trends by Year

### Count transactions & revenue per year

In [39]:
yearly_summary = trx_joined.groupBy("year").agg(
    F.count("*").alias("transaction_count"),
    F.sum("price").alias("total_revenue")
).orderBy("year")

yearly_summary.show()

+----+-----------------+----------------+
|year|transaction_count|   total_revenue|
+----+-----------------+----------------+
|2018|            88988|2644.69898305084|
|2019|           247090|6596.94828813579|
+----+-----------------+----------------+



### Top Product Groups by Year

In [40]:
top_groups_year = trx_joined.groupBy("year", "product_group_name") \
    .count() \
    .orderBy("year", F.desc("count"))

top_groups_year.show(20)

+----+-------------------+-----+
|year| product_group_name|count|
+----+-------------------+-----+
|2018| Garment Upper body|42389|
|2018| Garment Lower body|19516|
|2018|  Garment Full body| 6828|
|2018|          Underwear| 6617|
|2018|        Accessories| 5497|
|2018|     Socks & Tights| 2854|
|2018|              Shoes| 2116|
|2018|           Swimwear| 1750|
|2018|          Nightwear| 1339|
|2018|            Unknown|   58|
|2018|           Cosmetic|   13|
|2018|              Items|    5|
|2018|               Bags|    3|
|2018|Underwear/nightwear|    2|
|2018|   Interior textile|    1|
|2019| Garment Upper body|92730|
|2019| Garment Lower body|56301|
|2019|  Garment Full body|28806|
|2019|           Swimwear|26703|
|2019|          Underwear|18497|
+----+-------------------+-----+
only showing top 20 rows



### Top Departments by Year

In [41]:
top_depts_year = trx_joined.groupBy("year", "department_name") \
    .count() \
    .orderBy("year", F.desc("count"))

top_depts_year.show(20)

+----+-------------------+-----+
|year|    department_name|count|
+----+-------------------+-----+
|2018|           Knitwear| 7969|
|2018|            Trouser| 5237|
|2018|             Blouse| 4044|
|2018|Expressive Lingerie| 3451|
|2018|             Jersey| 3334|
|2018|       Jersey Basic| 3293|
|2018|      Tops Knitwear| 2701|
|2018|            Basic 1| 2683|
|2018|           Trousers| 2315|
|2018|       Jersey fancy| 2229|
|2018|     Denim Trousers| 2059|
|2018|  Tops Fancy Jersey| 2007|
|2018|            Outwear| 1655|
|2018|              Dress| 1617|
|2018|           Swimwear| 1599|
|2018|  Ladies Sport Bras| 1569|
|2018|       Tights basic| 1485|
|2018|            Dresses| 1372|
|2018|    Casual Lingerie| 1258|
|2018|         Tops Woven| 1181|
+----+-------------------+-----+
only showing top 20 rows



### Average Price by Year

In [42]:
avg_price_year = trx_joined.groupBy("year") \
    .agg(F.avg("price").alias("avg_price")) \
    .orderBy("year")

avg_price_year.show()

+----+--------------------+
|year|           avg_price|
+----+--------------------+
|2018|0.029719726064759745|
|2019| 0.02669856444265567|
+----+--------------------+



### Top Product Types per Year (extra detail)

In [43]:
top_types_year = trx_joined.groupBy("year", "product_type_name") \
    .count() \
    .orderBy("year", F.desc("count"))

top_types_year.show(20)

+----+-----------------+-----+
|year|product_type_name|count|
+----+-----------------+-----+
|2018|          Sweater|15738|
|2018|         Trousers|13882|
|2018|            Dress| 6225|
|2018|          T-shirt| 4869|
|2018|           Blouse| 4345|
|2018|              Top| 3600|
|2018|              Bra| 3227|
|2018| Underwear bottom| 2804|
|2018|            Shirt| 2598|
|2018|         Vest top| 2531|
|2018|  Leggings/Tights| 2486|
|2018|           Jacket| 2442|
|2018|            Skirt| 2371|
|2018|           Hoodie| 2201|
|2018|            Socks| 1680|
|2018|         Cardigan| 1502|
|2018|           Blazer| 1216|
|2018| Underwear Tights| 1172|
|2018|            Boots|  970|
|2018|            Scarf|  929|
+----+-----------------+-----+
only showing top 20 rows



# Task 2
We have to estimate the values for the full population using the 3% sample.

## Compute sample statistics

In [44]:
sample_revenue = trx_joined.agg(F.sum("price")).first()[0]
sample_revenue

9241.647271186823

In [45]:
sample_customers = trx_joined.select("customer_id").distinct().count()
sample_customers

230688

In [46]:
sample_transactions = trx_joined.count()
sample_transactions

336078

## Scale up to estimate full population

In [47]:
scale = 100 / 3
est_total_revenue = sample_revenue * scale
est_total_customers = sample_customers * scale
est_total_transactions = sample_transactions * scale

## Average yearly expenses per customer

In [48]:
avg_spend_per_customer = est_total_revenue / est_total_customers

In [49]:
(result_revenue,
 result_customers,
 result_transactions,
 result_avg_spend) = (est_total_revenue,
                      est_total_customers,
                      est_total_transactions,
                      avg_spend_per_customer)

result_revenue, result_customers, result_transactions, result_avg_spend

(308054.90903956076, 7689600.000000001, 11202600.0, 0.04006123973152839)

# Task 3 — Data Quality Report

All data quality checks were performed during the preprocessing stage.
Below I summarize the results:

### 1. Missing Values
- `transactions.csv`: No nulls in any column.
- `articles.csv`: Only `detail_desc` contains nulls (expected optional text field).

### 2. Duplicate Transactions
- Found 336,078 rows with 335,151 unique rows → 927 duplicates.
- Duplicates were exact copies and were removed to avoid double-counting.

### 3. Join Quality (transactions × articles)
- A very small number of transactions had no matching article metadata.
- These represent discontinued or corrupted product records.

### 4. Outlier Checks
- No negative prices.
- No unusual extreme values in price distribution.

### 5. Customer ID Consistency
- All customer IDs are present and uniformly formatted.

### 6. Article Metadata Consistency
- `article_id` is unique in articles.
- Hierarchical product attributes (type, department, group) are consistent and populated.

### 7. Date Range
- Dates span from 2018-09-20 to 2019-09-19.
- No future dates, no corrupted dates.

### ✔ Overall
The dataset is generally **high quality**, requiring minimal cleaning.
All detected issues were corrected (duplicate removal) or documented (missing metadata for a few article IDs).

# Task 4

In [57]:
trx_joined.groupBy("colour_group_name").count().orderBy(F.desc("count")).show(20)

+-----------------+------+
|colour_group_name| count|
+-----------------+------+
|            Black|116525|
|            White| 36624|
|        Dark Blue| 28495|
|      Light Beige| 11712|
|             Blue| 11686|
|       Light Blue| 10130|
|              Red|  9905|
|             Grey|  9396|
|       Light Pink|  8486|
|         Dark Red|  8483|
|        Off White|  8175|
|        Dark Grey|  7943|
|            Beige|  7687|
|   Greenish Khaki|  7657|
|       Dark Green|  6619|
|           Yellow|  4963|
|       Light Grey|  4602|
|             Pink|  4369|
|  Yellowish Brown|  3868|
|     Light Orange|  3854|
+-----------------+------+
only showing top 20 rows



In [58]:
trx_joined.groupBy("sales_channel_id").agg(
    F.count("*").alias("transactions"),
    F.avg("price").alias("avg_price")
).orderBy("sales_channel_id").show()

+----------------+------------+-------------------+
|sales_channel_id|transactions|          avg_price|
+----------------+------------+-------------------+
|               1|      103464|0.02296408068572467|
|               2|      232614|0.02951538440127752|
+----------------+------------+-------------------+



In [59]:
trx_joined.groupBy("season").agg(
    F.countDistinct("product_type_name").alias("unique_product_types")
).orderBy("season").show()

+------+--------------------+
|season|unique_product_types|
+------+--------------------+
|  Fall|                 101|
|Spring|                  96|
|Summer|                 100|
|Winter|                  97|
+------+--------------------+



In [60]:
premium_keywords = ["Premium", "Quality", "Denim", "Wool", "Knit", "Leather"]
basic_keywords = ["Basic", "Jersey Basic", "Basic 1"]

trx_segmented = trx_joined.withColumn(
    "segment",
    F.when(F.lower("department_name").rlike("|".join([k.lower() for k in premium_keywords])), "Premium")
     .when(F.lower("department_name").rlike("|".join([k.lower() for k in basic_keywords])), "Basic")
     .otherwise("Other")
)

trx_segmented.groupBy("season", "segment") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", "segment") \
    .show()

+------+-------+-----+
|season|segment|count|
+------+-------+-----+
|  Fall|  Basic| 9022|
|  Fall|  Other|54491|
|  Fall|Premium|16013|
|Spring|  Basic| 8828|
|Spring|  Other|70354|
|Spring|Premium| 6795|
|Summer|  Basic|10813|
|Summer|  Other|81339|
|Summer|Premium| 7070|
|Winter|  Basic| 7179|
|Winter|  Other|52198|
|Winter|Premium|11976|
+------+-------+-----+



In [61]:
basket_size = trx_joined.groupBy("customer_id").agg(
    F.count("*").alias("items_bought"),
    F.sum("price").alias("total_spent")
)

basket_size_summary = basket_size.agg(
    F.avg("items_bought").alias("avg_items_per_customer"),
    F.avg("total_spent").alias("avg_spending_per_customer")
)

basket_size_summary.show()

+----------------------+-------------------------+
|avg_items_per_customer|avg_spending_per_customer|
+----------------------+-------------------------+
|    1.4568508114856429|     0.040061239731527656|
+----------------------+-------------------------+



In [62]:
trx_joined.groupBy("prod_name") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

+--------------------+-----+
|           prod_name|count|
+--------------------+-----+
|      Luna skinny RW| 1483|
|Jade HW Skinny De...| 1255|
|           Tilly (1)|  941|
|Timeless Midrise ...|  897|
|     Kanta slacks RW|  877|
|      Jade Denim TRS|  808|
|           Despacito|  801|
|Skinny Ankle R.W ...|  794|
|      SUPREME tights|  771|
|               Gyda!|  770|
+--------------------+-----+
only showing top 10 rows



In [63]:
trx_joined.describe("price").show()

+-------+-------------------+
|summary|              price|
+-------+-------------------+
|  count|             336078|
|   mean| 0.0274985190080482|
| stddev|0.01921244097171475|
|    min|  1.864406779661E-4|
|    max| 0.5067796610169492|
+-------+-------------------+



In [64]:
monthly_rev = trx_joined.groupBy("year", "month") \
    .agg(F.sum("price").alias("monthly_revenue")) \
    .orderBy("year", "month")

monthly_rev.show()

+----+-----+------------------+
|year|month|   monthly_revenue|
+----+-----+------------------+
|2018|    9|362.12008474576174|
|2018|   10| 840.4624576271198|
|2018|   11| 788.9276101694908|
|2018|   12| 653.1888305084728|
|2019|    1| 674.2764745762685|
|2019|    2| 625.9581694915264|
|2019|    3| 751.4301016949117|
|2019|    4|  845.350338983048|
|2019|    5| 862.5826779660973|
|2019|    6| 970.3145254237215|
|2019|    7| 800.0118305084685|
|2019|    8| 616.7565254237264|
|2019|    9| 450.2676440677954|
+----+-----+------------------+



# Notes/Feedback
## Bootstraping for revenue
